# Lab 8.4 &mdash; Sequential Pipeline of Specialists

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Run the CS specialists in a fixed order over one customer ticket
- Feed each stage the previous stage's accumulated case
- See why a clean, ordered hand-off keeps each stage reliable

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; routing, synthesis, tool wiring, agent structure &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Sequential — a pipeline of specialists](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-08-04"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
The simplest collaboration is the **sequential pipeline** (deck slide 9): agents run in a **fixed
order**, each transforming the running case &mdash; for a support ticket that is **triage &rarr;
billing &rarr; tech**. It's a relay where the baton is the growing case. Each stage gets a **clean,
focused input** (the prior stage's output), so each does its narrow job well &mdash; and you can swap
any stage independently. (Watch out: errors **propagate** downstream, and it's **serial**, so latency
adds up.)

In [ ]:
# Each stage is a specialist that takes the running case and returns it, extended with its note.
print("pipeline: triage -> billing -> tech")

## Your Turn
Complete `run_pipeline` so each stage receives the previous stage's output.

In [ ]:
def run_pipeline(ticket, stages):
    case = ticket
    trail = []
    for stage in stages:
        case = ___   # TODO: run this stage on the CURRENT case (its input is the previous stage output)
        trail.append(case)
    return {"case": case, "trail": trail}

STAGES = [
    lambda c: c + " | triage -> billing+tech",
    lambda c: c + " | billing: duplicate charge on 4471",
    lambda c: c + " | tech: matches BUG-231",
]

try:
    out = run_pipeline('ticket: charged twice, app crashing', STAGES)
    print('final case:', out['case'])
    for step in out['trail']:
        print('  step:', step)
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the ticket passes through every stage in order", lambda: run_pipeline("t", STAGES)["case"].endswith("| tech: matches BUG-231"))
expect_true("one trail entry is recorded per stage", lambda: len(run_pipeline("t", STAGES)["trail"]) == 3)
expect_true("each stage builds on the previous stage's output", lambda: run_pipeline("t", STAGES)["trail"][1].startswith(run_pipeline("t", STAGES)["trail"][0]))
expect_true("triage runs before the specialists", lambda: "triage" in run_pipeline("t", STAGES)["trail"][0] and "billing" in run_pipeline("t", STAGES)["trail"][1])
expect_true("the final trail entry is the final case", lambda: run_pipeline("t", STAGES)["trail"][-1] == run_pipeline("t", STAGES)["case"])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
A pipeline is the multi-agent version of Module 7's automation pipeline -- each stage a specialist over the same ticket. Clean, ordered hand-offs make each stage reliable; just remember errors propagate downstream.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>